In [18]:
!pip install requests
!pip install dotenv

In [19]:
import requests
import json
import time
import re  # Regex for extracting Planner Plan ID from URLs
import urllib.parse
from dotenv import load_dotenv
import os



In [73]:
# Load environment variables
load_dotenv()
# Access token from .env
ACCESS_TOKEN = os.getenv("ACCESS_TOKEN")


headers = {
    "Authorization": f"Bearer {ACCESS_TOKEN}",
    "Accept": "application/json"
}

In [59]:
# Store results
channel_planners_detected = []

# Microsoft Graph API base URL
graph_base_url = "https://graph.microsoft.com/v1.0"

# Function to extract the real Plan ID from a Planner URL
def extract_plan_id_from_planner_url(planner_url: str) -> str:
    if not planner_url:
        return ""
    
    decoded_url = urllib.parse.unquote(planner_url)
    parsed = urllib.parse.urlparse(decoded_url)
    qs = urllib.parse.parse_qs(parsed.query)
    
    real_planner_link = qs["webUrl"][0] if "webUrl" in qs and qs["webUrl"] else decoded_url
    
    plan_id_match = re.search(r'/PlanViews/([^?&]+)', real_planner_link)
    if plan_id_match:
        return plan_id_match.group(1)
    
    plan_id_match = re.search(r'[?&]planId=([^&]+)', real_planner_link)
    if plan_id_match:
        return plan_id_match.group(1)
    
    return ""

# Function to clean "tt." formatted Planner IDs
def clean_plan_id(plan_id):
    if not plan_id:
        return None
    match = re.search(r"tt\.([a-zA-Z0-9-_]+)", plan_id)
    return f"tt.{match.group(1)}" if match else None

# Function to check if a tab is a valid Planner Tab
def is_planner_tab(tab_name, planner_id, planner_url):
    if planner_id and planner_id.startswith("tt."):
        return True
    if "planner" in tab_name.lower() or "tasks" in tab_name.lower():
        return True
    if planner_url and "tasks.office.com" in planner_url:
        return True
    return False

# Function to get all teams
def get_all_teams(headers):
    teams_url = f"{graph_base_url}/me/joinedTeams"
    response = requests.get(teams_url, headers=headers)
    if response.status_code == 200:
        return response.json().get("value", [])
    return []

# Function to get all channels for a given team
def get_team_channels(team_id, headers):
    channels_url = f"{graph_base_url}/teams/{team_id}/channels"
    response = requests.get(channels_url, headers=headers)
    if response.status_code == 200:
        return response.json().get("value", [])
    return []

# Function to get all tabs for a given channel
def get_channel_tabs(team_id, channel_id, headers):
    tabs_url = f"{graph_base_url}/teams/{team_id}/channels/{channel_id}/tabs"
    response = requests.get(tabs_url, headers=headers)
    if response.status_code == 200:
        return response.json().get("value", [])
    return []

# Function to split planner_tabs into separate objects
def split_planner_tabs(data):
    separated_items = []
    
    base_item = {
        "project_name": data["project_name"],
        "team_id": data["team_id"],
        "channel_name": data["channel_name"],
        "channel_id": data["channel_id"],
    }
    
    for tab in data["planner_tabs"]:
        new_item = base_item.copy()
        new_item["planner_tabs"] = [tab]  # Ensure it's a list with one element
        separated_items.append(new_item)
    
    return separated_items

In [61]:
# Fetch all teams
teams = get_all_teams(headers)

#print(teams)

for team in teams:
    team_id = team["id"]
    project_name = team.get("displayName", "Unknown Team")
    #print(f"\n🔍 Fetching channels for team: {project_name} ({team_id})")
    
    channels = get_team_channels(team_id, headers)
    
    for channel in channels:
        channel_name = channel["displayName"]
        channel_id = channel["id"]
        detected_planner_tabs = []
        
        tabs = get_channel_tabs(team_id, channel_id, headers)
        for tab in tabs:
            tab_name = tab.get("displayName", "")
            planner_id = tab.get("configuration", {}).get("entityId")
            planner_url = tab.get("webUrl")
            
            extracted_plan_id = extract_plan_id_from_planner_url(planner_url)
            if extracted_plan_id:
                planner_id = extracted_plan_id
            else:
                planner_id = clean_plan_id(planner_id)
            
            if planner_id and is_planner_tab(tab_name, planner_id, planner_url):
                detected_planner_tabs.append({
                    "tab_name": tab_name.encode("utf-8").decode("unicode_escape"),
                    "planner_id": planner_id,
                    "planner_url": planner_url,
                })
                #print(f"✅ Planner Tab Found: {tab_name} ({planner_id})")
        
        if detected_planner_tabs:
            channel_planners_detected.append({
                "project_name": project_name,
                "team_id": team_id,
                "channel_name": channel_name.encode("utf-8").decode("unicode_escape"),
                "channel_id": channel_id,
                "planner_tabs": detected_planner_tabs
            })
    
    time.sleep(1)  # Avoid hitting API rate limits

[{'id': '1d092b0f-f8eb-459c-b391-f4487e66680f', 'createdDateTime': None, 'displayName': 'LIDL', 'description': 'LIDL', 'internalId': None, 'classification': None, 'specialization': None, 'visibility': None, 'webUrl': None, 'isArchived': False, 'tenantId': '15dd7c6b-9ed1-4bf4-b2be-c602b1abd5e7', 'isMembershipLimitedToOwners': None, 'memberSettings': None, 'guestSettings': None, 'messagingSettings': None, 'funSettings': None, 'discoverySettings': None, 'tagSettings': None, 'summary': None}, {'id': 'd3a1f29a-40aa-4e27-b286-21f7dc5f3f7b', 'createdDateTime': None, 'displayName': 'ACTIV GROUP', 'description': 'ACTIV GROUP', 'internalId': None, 'classification': None, 'specialization': None, 'visibility': None, 'webUrl': None, 'isArchived': False, 'tenantId': '15dd7c6b-9ed1-4bf4-b2be-c602b1abd5e7', 'isMembershipLimitedToOwners': None, 'memberSettings': None, 'guestSettings': None, 'messagingSettings': None, 'funSettings': None, 'discoverySettings': None, 'tagSettings': None, 'summary': None},

In [62]:
# Split and save planner tabs
all_split_data = []
for item in channel_planners_detected:
    all_split_data.extend(split_planner_tabs(item))

# Convert back to JSON string for output
output_json = json.dumps(all_split_data, indent=4, ensure_ascii=False)

# #print or save the output
#print(output_json)

# Optional: Save the output to a new file
with open("teams_data.json", "w", encoding="utf-8") as output_file:
    output_file.write(output_json)


[
    {
        "project_name": "ACTIV GROUP",
        "team_id": "d3a1f29a-40aa-4e27-b286-21f7dc5f3f7b",
        "channel_name": "General",
        "channel_id": "19:0020527fe41e4a35902ca528b6ca31d3@thread.skype",
        "planner_tabs": [
            {
                "tab_name": "Tasks ACTIV GROUP TERRASSA",
                "planner_id": "QqtnCj433E-tc_EuxF0YHZcAGsto",
                "planner_url": "https://teams.microsoft.com/l/entity/com.microsoft.teamspace.tab.planner/_djb2_msteams_prefix_2009202136?webUrl=https%3a%2f%2ftasks.office.com%2f15dd7c6b-9ed1-4bf4-b2be-c602b1abd5e7%2fHome%2fPlanViews%2fQqtnCj433E-tc_EuxF0YHZcAGsto%3fType%3dPlanLink%26Channel%3dTeamsTab&label=Tasks+ACTIV+GROUP+TERRASSA&context=%7b%0d%0a++%22canvasUrl%22%3a+%22https%3a%2f%2ftasks.teams.microsoft.com%2fteamsui%2f%7btid%7d%2fHome%2fPlannerFrame%3fpage%3d7%26auth_pvr%3dOrgId%26auth_upn%3d%7buserPrincipalName%7d%26groupId%3d%7bgroupId%7d%26planId%3dQqtnCj433E-tc_EuxF0YHZcAGsto%26channelId%3d%7bchannelId%7d%2

In [74]:
team_id = "a1cdf9e3-a48c-4385-93e1-39d4fb66af65"  # PRIMAFRIO EXT


In [78]:
# Fetch all channels for the team
channels = get_team_channels(team_id, headers)
channels.sort(key=lambda x: x["displayName"])
teams_data = []

for channel in channels:
    channel_name = channel["displayName"]
    channel_id = channel["id"]
    
    #print(f"\n🔍 Fetching tabs for channel: {channel_name} ({channel_id})")
    tabs = get_channel_tabs(team_id, channel_id, headers)
    
    channel_info = {
        "channel_name": channel_name,
        "channel_id": channel_id,
        "tabs": []
    }
    
    for tab in tabs:
        tab_info = {
            "tab_name": tab.get("displayName", ""),
            "tab_id": tab.get("id", ""),
            "web_url": tab.get("webUrl", ""),
            "raw_tab_id": tab.get("configuration", {}).get("entityId", "")  # Extract raw tab ID
        }
        channel_info["tabs"].append(tab_info)
        #print(f"✅ Tab Found: {tab_info['tab_name']} ({tab_info['tab_id']})")
    
    teams_data.append(channel_info)

# Convert to JSON and save
output_json = json.dumps(teams_data, indent=4, ensure_ascii=False)
#print(output_json)



🔍 Fetching tabs for channel: 01.- VILAMALLA 2111B (19:63e1342eedb643cb81ec9c163a91a036@thread.tacv2)
✅ Tab Found: Wiki (74b28430-bef5-4788-9647-04e743d345e0)
✅ Tab Found: Notes VILAMALLA Primafrio (6cd7efa9-84ae-4182-9f21-9ad45bc10cb1)
✅ Tab Found: Project VILAMALLA Primafrio (24912d5a-3dcb-4085-bf16-3fe3bfa9e35a)
✅ Tab Found: Files (3ed5b337-c2c9-4d5d-b7b4-84ff09a8fc1c)

🔍 Fetching tabs for channel: 02.- SAN ANTONIO BENAGEBER_2111A (19:821f29f6c07b4504ba57e12a443c0538@thread.tacv2)
✅ Tab Found: Wiki (02e49ecb-fd68-4468-9dba-fd881c3fa0fd)
✅ Tab Found: Notes SAN ANTONIO Primafrio (b4d28e9d-be7b-46e9-8b53-5ca27047b842)
✅ Tab Found: Project SANT ANTONIO (5986aff2-0b18-4494-b4a6-49d0a5721255)
✅ Tab Found: Tareas ONDINA Sant Antonio (0bfe2e1e-c7f9-4855-b90a-249cadf775c0)
✅ Tab Found: Files (3ed5b337-c2c9-4d5d-b7b4-84ff09a8fc1c)

🔍 Fetching tabs for channel: 03.- PINTO 2 2304B (19:17fb409a5898487c8c99cdcb14cc6caf@thread.tacv2)
✅ Tab Found: Notes PINTO Primafrio (c8774248-fb53-4edb-b77d-2c1e